# 🚀 Tái Lập Thí Nghiệm LiDAR (Bảng 2) trên Google Colab## Mô hình: Stable Diffusion v1.5 + LiDAR (DPM-5 / $n=50$, DDIM 50 bước)Notebook này được tối ưu hóa đặc biệt dành riêng cho **Google Colab (GPU T4 miễn phí)**:1. **Lưu trữ trực tiếp thời gian thực vào Google Drive**: Mỗi prompt sinh xong sẽ được ghi thẳng vào Google Drive của bạn. Dù Colab bị ngắt kết nối hay hết giờ (sau 4 tiếng), **100% dữ liệu không bao giờ bị mất**.2. **Cơ chế `--resume` thông minh**: Khi mở phiên Colab mới, chương trình sẽ tự động đọc lại Google Drive, bỏ qua các prompt đã làm xong và chạy nối tiếp ngay lập tức.3. **Cấu hình siêu tốc (DDIM 50 bước)**: Hoàn thành toàn bộ 553 prompt trong **~3.0 giờ** (vừa vặn trong 1 phiên Colab duy nhất).

## 1. Kiểm tra GPU Google Colab

In [ ]:
import os, sys, torchprint(f"Phiên bản Python: {sys.version}")print(f"Phiên bản PyTorch: {torch.__version__}")print(f"Hỗ trợ CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")    print(f"Bộ nhớ VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")!nvidia-smi

## 2. Gắn Kết Google Drive (Để Lưu Trữ Vĩnh Viễn & Chạy Nối Tiếp)Khi chạy ô này, Google Colab sẽ hỏi quyền truy cập Google Drive. Toàn bộ checkpoint và ảnh sinh ra sẽ được tự động lưu vào thư mục `My Drive/LiDAR_Experiment`.

In [ ]:
# Gắn kết Google Drivefrom google.colab import driveimport os, shutil, globdrive.mount('/content/drive')# Thư mục lưu trữ vĩnh viễn trên Google Drive của bạnDRIVE_DIR = "/content/drive/MyDrive/LiDAR_Experiment"os.makedirs(f"{DRIVE_DIR}/Lookahead_samples", exist_ok=True)os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)# Tự động quét và giải nén file zip dữ liệu cũ nếu có upload lên Drive hoặc Colabdrive_zips = glob.glob(f"{DRIVE_DIR}/*.zip") + glob.glob("/content/*.zip")for zf in drive_zips:    print(f"📦 Tìm thấy file zip dữ liệu: {zf}. Đang giải nén vào Google Drive...")    try:        shutil.unpack_archive(zf, DRIVE_DIR)        print(f"✅ Đã giải nén xong: {zf}")    except Exception as e:        print(f"⚠️ Lỗi giải nén: {e}")print(f"✅ Đã kết nối Google Drive! Thư mục lưu trữ: {DRIVE_DIR}")

## 3. Thiết lập Mã Nguồn & Cài đặt Thư viện

In [ ]:
# Tải mã nguồn RS-LiDAR về Colabimport osWORKDIR = "/content/RS-LiDAR"if not os.path.exists(WORKDIR):    !git clone https://github.com/leekwanreal/RS-LiDAR.git {WORKDIR}else:    !cd {WORKDIR} && git pull origin mainos.chdir(WORKDIR)%cd {WORKDIR}# Tạo liên kết tượng trưng (Symlink) để lưu thẳng vào Google Driveif not os.path.exists(f"{WORKDIR}/Lookahead_samples"):    !ln -s "{DRIVE_DIR}/Lookahead_samples" "{WORKDIR}/Lookahead_samples"if not os.path.exists(f"{WORKDIR}/Target_samples"):    !ln -s "{DRIVE_DIR}/Target_samples" "{WORKDIR}/Target_samples"# Cài đặt các thư viện phụ thuộc đã kiểm chứng!pip install -q --upgrade protobuf!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft!pip install -q git+https://github.com/openai/CLIP.git!pip install -q git+https://github.com/THUDM/ImageReward.git!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas# Tải file từ điển BPE cho HPSv2import urllib.request, hpsv2hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)if not os.path.exists(hpsv2_vocab):    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)n_look = len(glob.glob(f"{DRIVE_DIR}/Lookahead_samples/*/[0-9]*"))n_targ = len(glob.glob(f"{DRIVE_DIR}/Target_samples/*/[0-9]*"))print(f"✅ Môi trường Colab đã sẵn sàng! Đã tìm thấy trên Drive: {n_look} Lookahead prompts, {n_targ} Target prompts.")

## 4. Cấu hình Siêu Tham Số Thí Nghiệm (Thiết lập theo Bảng 2)Cấu hình chuẩn tái lập dòng **SD v1.5 + LiDAR (DPM-5 / $n=50$, DDIM-50)**:- **Phase 1 (Lookahead)**: DPM-Solver 5 bước (`--num_inference_steps=5`), tạo $n=50$ hạt (`--num_particles=50`).- **Phase 2 (Target Sampling)**: DDIM 50 bước (`--num_inference_steps=50`, `--eta=0.0`), $N=4$ ảnh/prompt.- **Guidance Parameters**: $s=12.5$, $\lambda=5000$, $T_{end}=200$ (ngắt guidance ở 20% denoising cuối).

In [ ]:
# ==================== CẤU HÌNH SIÊU THAM SỐ ====================SEED = 100                       # Random seedNUM_LOOKAHEAD_PARTICLES = 50     # Số hạt lookahead n = 50LOOKAHEAD_STEPS = 5              # Số bước DPM-Solver = 5 (DPM-5)LOOKAHEAD_TAG = f"{SEED}_{NUM_LOOKAHEAD_PARTICLES}_{LOOKAHEAD_STEPS}"# Tham số lấy mẫu đích Phase 2 (SD v1.5 với DDIM 50 bước)MODEL_NAME = "runwayml/stable-diffusion-v1-5"NUM_TARGET_STEPS = 50            # 50 bước cho DDIMETA = 0.0                        # eta = 0.0 cho DDIMTARGET_PARTICLES = 4             # 4 ảnh trên mỗi prompt (chuẩn đánh giá GenEval)SCALE = 12.5                     # Hệ số guidance s = 12.5 cho SD v1.5LAMBDA = 5000                    # Hệ số nhiệt độ lambda = 5000RESAMPLE_T_END = 200             # Ngưỡng kết thúc guidance sớm [1.0, 0.2]TOP_K = 50                       # Chọn top-k lookaheads (50)# Dữ liệu Prompt và Giới hạn số lượngPROMPT_FILE = "prompt_files/geneval_metadata.jsonl"MAX_PROMPTS = 553RUN_NAME = f"LiDAR_SD15_DPM5_n50_DDIM50_seed{SEED}"print(f"Tên lượt chạy: {RUN_NAME}")print(f"Cấu hình Target: DDIM {NUM_TARGET_STEPS} bước (eta={ETA})")print(f"Đường dẫn Lookahead: {LOOKAHEAD_TAG}")print(f"Tổng số Prompt cần xử lý: {MAX_PROMPTS}")

## 5. Giai Đoạn 1 (Phase 1): Lấy Mẫu Lookahead & Đánh Giá Reward (~1.5 giờ)Sinh $n=50$ hạt cho mỗi prompt bằng 5 bước DPM-Solver, lưu latent và kết quả trực tiếp vào Google Drive.

In [ ]:
# Thực thi Giai đoạn 1: Lookahead Samplingimport osos.environ["USE_TF"] = "0"os.environ["USE_TORCH"] = "1"os.chdir(WORKDIR)lookahead_cmd = f"""cd {WORKDIR} && USE_TF=0 USE_TORCH=1 python lookahead_sampling.py \    --seed={SEED} \    --num_particles={NUM_LOOKAHEAD_PARTICLES} \    --num_inference_steps={LOOKAHEAD_STEPS} \    --model_name="{MODEL_NAME}" \    --guidance_reward_fn="ImageReward" \    --metrics_to_compute="ImageReward#Clip-Score" \    --prompt_path="{PROMPT_FILE}" \    --max_prompt={MAX_PROMPTS} \    --resume"""print(">>> Đang bắt đầu Giai đoạn 1 (Lookahead Sampling)...")!{lookahead_cmd}

## 6. Giai Đoạn 2 (Phase 2): Lấy Mẫu Đích LiDAR Sampling (~3.0 giờ)Sử dụng các hạt Lookahead từ Giai đoạn 1 để hướng dẫn quá trình sinh ảnh 50 bước DDIM ($N=4$ ảnh/prompt).

In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
from IPython.display import display

target_dir = f"{DRIVE_DIR}/Target_samples/{RUN_NAME}"
result_files = sorted(glob.glob(f"{target_dir}/[0-9]*/results.json"))

if result_files:
    print(f"📊 Đang tổng hợp chỉ số đánh giá từ {len(result_files)} prompt...")
    metric_keys = ["ImageReward", "Clip-Score", "HumanPreference", "Clip-Diversity", "AS"]
    collected_means = {k: [] for k in metric_keys}
    
    for rf in result_files:
        try:
            with open(rf, "r") as f:
                res = json.load(f)
            for k in metric_keys:
                if k in res and "mean" in res[k]:
                    collected_means[k].append(res[k]["mean"])
        except Exception:
            pass
    
    final_metrics = {}
    for k, vals in collected_means.items():
        if vals:
            final_metrics[k] = {
                "mean": float(np.mean(vals)),
                "std": float(np.std(vals)),
                "min": float(np.min(vals)),
                "max": float(np.max(vals)),
            }
    
    with open(f"{target_dir}/final_metrics.json", "w") as f:
        json.dump(final_metrics, f, indent=4)
    
    ir_val = final_metrics.get('ImageReward', {}).get('mean', 0.0)
    clip_val = final_metrics.get('Clip-Score', {}).get('mean', 0.0)
    hps_val = final_metrics.get('HumanPreference', {}).get('mean', 0.0)
    div_val = final_metrics.get('Clip-Diversity', {}).get('mean', 0.0)
    as_val = final_metrics.get('AS', {}).get('mean', 0.0)
    
    table_data = [
        {
            "Phương Pháp": "SD v1.5 Gốc (Chưa lái)",
            "Số bước": "50 DDIM",
            "ImageReward ↑": "-0.076",
            "CLIP-Score ↑": "0.264",
            "HPS v2.1 ↑": "0.252",
            "Độ Phù Hợp": "Baseline"
        },
        {
            "Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDIM-50)",
            "Số bước": "50 DDIM",
            "ImageReward ↑": "0.378",
            "CLIP-Score ↑": "0.278",
            "HPS v2.1 ↑": "0.277",
            "Độ Phù Hợp": "Target Benchmark"
        },
        {
            "Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDPM-100)",
            "Số bước": "100 DDPM",
            "ImageReward ↑": "0.384",
            "CLIP-Score ↑": "0.278",
            "HPS v2.1 ↑": "0.276",
            "Độ Phù Hợp": "Upper Bound"
        },
        {
            "Phương Pháp": "🔥 KẾT QUẢ CHẠY THỰC TẾ (Ours)",
            "Số bước": f"{NUM_TARGET_STEPS} {'DDIM' if ETA==0.0 else 'DDPM'}",
            "ImageReward ↑": f"{ir_val:.4f}",
            "CLIP-Score ↑": f"{clip_val:.4f}",
            "HPS v2.1 ↑": f"{hps_val:.4f}",
            "Độ Phù Hợp": f"Δ IR: {ir_val - 0.378:+.3f}"
        }
    ]
    
    df = pd.DataFrame(table_data)
    print("\n======================= 📊 BẢNG ĐỐI CHIẾU KẾT QUẢ VỚI BÀI BÁO =======================")
    display(df)
    
    print("\n📈 Thống Kê Bổ Sung Từ Thực Nghiệm:")
    print(f" - Số lượng prompt đã hoàn thành: {len(result_files)}/{MAX_PROMPTS}")
    print(f" - ImageReward Mean: {ir_val:.4f}")
    print(f" - CLIP Score Mean: {clip_val:.4f}")
    print(f" - HPS v2.1 Mean: {hps_val:.4f}")
    print(f" - CLIP Diversity (Độ đa dạng ảnh): {div_val:.4f}")
    print(f" - Aesthetic Score (Điểm thẩm mỹ): {as_val:.4f}")
else:
    print(f"Chưa tìm thấy file kết quả tại {target_dir}. Vui lòng chạy Giai đoạn 2 trước.")


## 7. Đánh Giá Định Lượng & So Sánh Chi Tiết với Bảng 2Đọc kết quả từ file `final_metrics.json` và lập bảng đối chiếu trực tiếp với các mốc chuẩn trong Bảng 2 của bài báo.

In [ ]:
import jsonimport pandas as pdfrom IPython.display import displayresults_file = f"{DRIVE_DIR}/Target_samples/{RUN_NAME}/final_metrics.json"if os.path.exists(results_file):    with open(results_file, "r") as f:        metrics = json.load(f)        ir_val = metrics.get('ImageReward', {}).get('mean', 0.0)    clip_val = metrics.get('Clip-Score', {}).get('mean', 0.0)    hps_val = metrics.get('HumanPreference', {}).get('mean', 0.0)    div_val = metrics.get('Clip-Diversity', {}).get('mean', 0.0)    as_val = metrics.get('AS', {}).get('mean', 0.0)        table_data = [        {            "Phương Pháp": "SD v1.5 Gốc (Chưa lái)",            "Số bước": "50 DDIM",            "ImageReward ↑": "-0.076",            "CLIP-Score ↑": "0.264",            "HPS v2.1 ↑": "0.252",            "Độ Phù Hợp": "Baseline"        },        {            "Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDIM-50)",            "Số bước": "50 DDIM",            "ImageReward ↑": "0.378",            "CLIP-Score ↑": "0.278",            "HPS v2.1 ↑": "0.277",            "Độ Phù Hợp": "Target Benchmark"        },        {            "Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDPM-100)",            "Số bước": "100 DDPM",            "ImageReward ↑": "0.384",            "CLIP-Score ↑": "0.278",            "HPS v2.1 ↑": "0.276",            "Độ Phù Hợp": "Upper Bound"        },        {            "Phương Pháp": "🔥 KẾT QUẢ CHẠY THỰC TẾ (Ours)",            "Số bước": f"{NUM_TARGET_STEPS} {'DDIM' if ETA==0.0 else 'DDPM'}",            "ImageReward ↑": f"{ir_val:.4f}",            "CLIP-Score ↑": f"{clip_val:.4f}",            "HPS v2.1 ↑": f"{hps_val:.4f}",            "Độ Phù Hợp": f"Δ IR: {ir_val - 0.378:+.3f}"        }    ]        df = pd.DataFrame(table_data)    print("======================= 📊 BẢNG ĐỐI CHIẾU KẾT QUẢ VỚI BÀI BÁO =======================")    display(df)        print("📈 Thống Kê Bổ Sung Từ Thực Nghiệm:")    print(f" - CLIP Diversity (Độ đa dạng ảnh): {div_val:.4f}")    print(f" - Aesthetic Score (Điểm thẩm mỹ): {as_val:.4f}")else:    print(f"Chưa tìm thấy file kết quả tại {results_file}. Vui lòng chạy Giai đoạn 2 trước.")

## 8. Trực Quan Hóa Các Ảnh Mẫu Đã SinhHiển thị lưới hình ảnh 4 ảnh/prompt để kiểm tra trực quan chất lượng sinh ảnh.

In [ ]:
import globfrom PIL import Imageimport matplotlib.pyplot as plttarget_dir = f"{DRIVE_DIR}/Target_samples/{RUN_NAME}"grid_images = sorted(glob.glob(f"{target_dir}/*/grid.png"))if grid_images:    print(f"Tìm thấy {len(grid_images)} lưới ảnh prompt. Đang hiển thị 3 prompt đầu tiên:")    for img_path in grid_images[:3]:        img = Image.open(img_path)        plt.figure(figsize=(16, 4))        plt.imshow(img)        plt.axis('off')        plt.title(f"Chỉ số Prompt: {os.path.basename(os.path.dirname(img_path))}")        plt.show()else:    print("Chưa tìm thấy ảnh lưới mẫu nào.")